In [0]:
import requests
import datetime
import os

# 1. Configuration
year = 2026
month = 3
day = 3
hour = 9

specific_date = datetime.datetime(year, month, day, hour) # Year, month, day are required
date_hour = specific_date.strftime("%Y-%m-%d-%-H")

gh_url = f"https://data.gharchive.org/{date_hour}.json.gz"

# This must be a path to a Unity Catalog Volume
volume_path = "/Volumes/aws-s3-gh/default/aws-s3-gh-volumn-raw"
full_target_path = f"{volume_path}/{date_hour}.json.gz"

# 2. Ensure the directory exists (CRITICAL for Volumes)
# Serverless requires the folder structure to exist before writing
os.makedirs(volume_path, exist_ok=True)

# 3. Stream the download directly to the Volume
# Using 'stream=True' and shutil is memory-efficient for Serverless
print(f"Streaming {date_hour} to Volume...")
with requests.get(gh_url, stream=True) as r:
    r.raise_for_status()
    with open(full_target_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024): # 1MB chunks
            if chunk:
                f.write(chunk)

print(f"✅ Data landed in Volume: {full_target_path}")

In [0]:
import requests
import os
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

# --- 1. CONFIGURATION ---
start_date = datetime(2026, 3, 1)  # Your start range
end_date = datetime(2026, 3, 2)    # Your end range
volume_path = "/Volumes/aws-s3-gh/default/aws-s3-gh-volumn-raw"
max_workers = 8  # Number of files to download at the same time

# --- 2. FORMATTING HELPER ---
def get_gh_url(dt):
    # GH Archive uses single digits for month, day, and hour
    # %-m, %-d, %-H works on Databricks/Linux to remove leading zeros
    date_str = dt.strftime("%Y-%m-%d-%-H")
    return f"https://data.gharchive.org/{date_str}.json.gz", f"{date_str}.json.gz"

# --- 3. DOWNLOAD FUNCTION (With Retries) ---
def download_file(dt):
    url, filename = get_gh_url(dt)
    target_path = os.path.join(volume_path, filename)
    
    if os.path.exists(target_path):
        return f"⏩ Skipped: {filename}"

    try:
        # 3 Retries for robustness
        for attempt in range(3):
            with requests.get(url, stream=True, timeout=15) as r:
                if r.status_code == 200:
                    with open(target_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024):
                            f.write(chunk)
                    return f"✅ Downloaded: {filename}"
                elif r.status_code == 404:
                    return f"⚠️ 404 Not Found: {filename}"
        return f"❌ Failed after retries: {filename}"
    except Exception as e:
        return f"💥 Error: {filename} -> {str(e)}"

# --- 4. EXECUTION ---
# Create a list of all hours to download
date_list = []
curr = start_date
while curr <= end_date:
    date_list.append(curr)
    curr += timedelta(hours=1)

print(f"🚀 Starting download of {len(date_list)} files...")

# Run in parallel
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    results = list(executor.map(download_file, date_list))

# Summary
for res in results[-10:]: # Print last 10 results
    print(res)

In [0]:
# 1. Configuration - Using a fresh checkpoint to ensure a clean start
table_checkpoint = "/Volumes/aws-s3-gh/default/aws-s3-gh-volumn-raw/_checkpoints/github_events"
target_table = "`aws-s3-gh`.default.github_events"

# 2. Define the Stream
# We reuse the 'df' you already defined, or redefine it here for safety
streaming_df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("pathGlobFilter", "*.json.gz")
  .option("cloudFiles.schemaLocation", f"{table_checkpoint}/schema")
  .load("/Volumes/aws-s3-gh/default/aws-s3-gh-volumn-raw/"))

# 3. Write to the Table
# .awaitTermination() is key here to make the notebook wait for the data
query = (streaming_df.writeStream
  .option("checkpointLocation", f"{table_checkpoint}/data")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .toTable(target_table))

query.awaitTermination()

# 4. Final verification
print(f"Ingestion complete. Current row count:")
display(spark.sql(f"SELECT count(*) FROM {target_table}"))